# 05_训练小分类器

**学习目标：** 在一分钟级小例子中看懂训练循环，保存并重新加载模型。

适合：读过CS50P但基础不熟、刚开始深度学习。按顺序学习；每课先看讲解、跑示例、做练习，再查答案。


## 本课要点

本课使用合成的40维数字输入和40类标签，没有动作图像、不下载数据。目的仅是观察训练机制，分数不能作为比赛效果。logits是40个未归一化分数，交叉熵直接接收logits和整数标签。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/learning.py').is_file() and (p / 'config.json').is_file())
sys.path.insert(0, str(ROOT / 'tools'))
import learning as L
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)


## 构造可学习的教学数据


In [ ]:
get_ipython().run_line_magic('matplotlib', 'inline')
import torch
from torch import nn
import matplotlib.pyplot as plt
torch.set_num_threads(4)
torch.manual_seed(42)
y = torch.arange(40).repeat(4)
x = torch.nn.functional.one_hot(y, 40).float() + 0.03 * torch.randn(len(y), 40)
model = nn.Linear(40, 40)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.1)
print(x.shape, y.shape, model(x).shape)


## 一次循环中的先后顺序


In [ ]:
losses = []
before = model.weight.detach().clone()
model.train()
for step in range(60):
    optimizer.zero_grad()
    logits = model(x)
    loss = nn.functional.cross_entropy(logits, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
assert not torch.equal(before, model.weight)
print('首末loss', round(losses[0], 4), round(losses[-1], 4))
plt.plot(losses)
plt.xlabel('step'); plt.ylabel('training loss'); plt.show()


## 新对象加载本地参数


In [ ]:
model.eval()
with torch.inference_mode():
    expected = model(x[:2]).clone()
weight_path = OUT / 'toy-classifier.pt'
torch.save(model.state_dict(), weight_path)
restored = nn.Linear(40, 40)
restored.load_state_dict(torch.load(weight_path, map_location='cpu', weights_only=True))
restored.eval()
with torch.inference_mode():
    torch.testing.assert_close(restored(x[:2]), expected)
print('重载前后输出一致；仅教学模型：', weight_path)


## 动手练习

使用restored预测前两个样本，沿类别维选最大分数，得到长度2的类别编号。

显示“待完成”表示练习尚未作答；代码示例运行成功不等于练习完成。


In [ ]:
your_answer = None  # TODO：with inference_mode，调用argmax
if your_answer is None:
    print('待完成：类别轴是dim=1')
else:
    assert your_answer.shape == (2,)
    assert torch.equal(your_answer, y[:2])
    print('练习通过')


## 常见误区

eval控制Dropout/BatchNorm行为，inference_mode关闭梯度跟踪，二者作用不同。


## 过关检查与选读

逐句解释训练循环；重启后能加载文件；知道训练loss下降不保证泛化。

选做：改变示例中一个参数，先预测现象再运行，并记下原因。

[对应参考答案](../参考答案/05_训练小分类器.ipynb) · [官方选读](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

完成后在README.md#学习记录写下一句自己的解释。
